# Search Demo

Load the local FAISS bundle and run a simple CLIP text query against the current keyframe metadata.

In [ ]:
import json
from pathlib import Path

import faiss
import numpy as np

from backend.config import FAISS_INDEX_PATH, FAISS_METADATA_PATH
from backend.embedding.clip_encoder import encode_text

In [ ]:
index = faiss.read_index(str(FAISS_INDEX_PATH))
with open(FAISS_METADATA_PATH, encoding='utf-8') as f:
    metadata = json.load(f)

print(f'Loaded {index.ntotal} vectors and {len(metadata)} metadata rows.')

In [ ]:
query_text = 'a photo of a tree'
query_vec = encode_text(query_text).reshape(1, -1).astype(np.float32)
faiss.normalize_L2(query_vec)

top_k = 10
scores, indices = index.search(query_vec, top_k)

print(f"Query: {query_text}")
for rank, (score, idx) in enumerate(zip(scores[0], indices[0]), start=1):
    if idx < 0 or idx >= len(metadata):
        continue
    item = metadata[idx]
    print(f"#{rank:02d} | score={score:.4f} | {item.get('video_id','')} | frame={item.get('frame_id','')} | pts={item.get('pts_time', 0.0):.2f}s")

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

n = min(top_k, 10)
fig, axes = plt.subplots(2, 5, figsize=(20, 8))
fig.suptitle(f"Search results: {query_text}")

for i, ax in enumerate(axes.flat):
    if i >= n:
        ax.axis('off')
        continue
    idx = indices[0][i]
    if idx < 0 or idx >= len(metadata):
        ax.axis('off')
        continue
    item = metadata[idx]
    img_path = Path(item.get('path', ''))
    if img_path.exists():
        ax.imshow(Image.open(img_path).convert('RGB'))
    else:
        ax.text(0.5, 0.5, 'Image not found', ha='center', va='center', transform=ax.transAxes)
    ax.set_title(f"#{i+1} | {item.get('video_id','')} | {item.get('frame_id','')}")
    ax.axis('off')

plt.tight_layout()
plt.show()